# 🚀 TransLSTM-Predictor
**CNN-BiLSTM-Transformer Hybrid Stock Prediction System**

This notebook runs the full pipeline of the TransLSTM-Predictor in a Google Colab GPU environment.

---

## Pipeline
1. Environment Setup & Dependency Installation
2. Data Upload or Sample Data Generation
3. Settings (Hyperparameters)
4. Run Full Pipeline
   - Feature Engineering
   - Walk-forward Validation + Ensemble Training
   - Backtesting
   - Future Prediction

## 1. Environment Setup

In [ ]:
# Check GPU
!nvidia-smi

import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Clone repository (first run only)
import os
if not os.path.exists('TransLSTM-Predictor'):
    !git clone https://github.com/irhdab/TransLSTM-Predictor.git

%cd TransLSTM-Predictor
%pip install -U -q -r requirements.txt

## 2. Data Preparation

**Option A**: Upload CSV file directly  
**Option B**: Generate auto sample data in the cell below

In [ ]:
# Option A: File upload
# from google.colab import files
# uploaded = files.upload()  # Requires columns: date, open, high, low, close, volume
# CSV_PATH = list(uploaded.keys())[0]

# Option B: Generate sample data (S&P 500 style simulation)
import numpy as np
import pandas as pd

np.random.seed(42)
days = 1000
dates = pd.bdate_range(start='2020-01-02', periods=days)

# Generate prices based on random walk
returns = np.random.normal(0.0005, 0.015, days)
close = 100 * np.exp(np.cumsum(returns))
high = close * (1 + np.abs(np.random.normal(0, 0.008, days)))
low = close * (1 - np.abs(np.random.normal(0, 0.008, days)))
open_price = close * (1 + np.random.normal(0, 0.003, days))
volume = np.random.randint(1_000_000, 50_000_000, days)

df = pd.DataFrame({
    'date': dates,
    'open': open_price,
    'high': high,
    'low': low,
    'close': close,
    'volume': volume
})

os.makedirs('data', exist_ok=True)
CSV_PATH = 'data/sample_stock.csv'
df.to_csv(CSV_PATH, index=False)
print(f"✓ Sample data generation complete: {df.shape}")
df.tail()

## 3. Settings (Hyperparameters)

In [ ]:
import config.config as config

# === Modify values here as desired ===
config.EPOCHS = 50            # Reduce in Colab for quick testing
config.ENSEMBLE_SIZE = 3
config.WALK_FORWARD_FOLDS = 3
config.SEQ_LENGTH = 60
config.FUTURE_DAYS = 30
config.RANDOM_SEED = 42

config.validate()

print("=== Current Configuration ===")
print(f"  EPOCHS:             {config.EPOCHS}")
print(f"  ENSEMBLE_SIZE:      {config.ENSEMBLE_SIZE}")
print(f"  WALK_FORWARD_FOLDS: {config.WALK_FORWARD_FOLDS}")
print(f"  SEQ_LENGTH:         {config.SEQ_LENGTH}")
print(f"  FUTURE_DAYS:        {config.FUTURE_DAYS}")
print(f"  PREDICT_RETURNS:    {config.PREDICT_RETURNS}")
print(f"  TRANSACTION_COST:   {config.TRANSACTION_COST}")


## 4. Run Pipeline

In [ ]:
# Data Loading via shared pipeline (fixes train-only outlier bounds + date alignment)
from main import StockPipeline

pipeline = StockPipeline(csv_path=CSV_PATH)
pipeline.prepare_data()

# Expose aligned artifacts for inspection (post-indicator, no 21-day offset)
dp = pipeline.data_processor
data = pipeline.processed_data
features = pipeline.features
print(f"Features: {features.shape}, dates: {len(pipeline.aligned_dates)}")


In [ ]:
# Walk-forward Validation + Ensemble Training (shared implementation)
pipeline.run_walk_forward_validation()

# Expose last-fold results for the cells below
final_ensemble_models = pipeline.final_ensemble_models
final_scaler = pipeline.final_scalers['scaler']
final_target_scaler = pipeline.final_scalers['target_scaler']
final_test_actual = pipeline.last_fold_results['actual']
final_test_pred = pipeline.last_fold_results['pred']
final_test_dates = pipeline.last_fold_results['dates']
print(f"Last fold test samples: {len(final_test_dates)}")


## 5. Backtesting

In [ ]:
backtester = Backtester(config)
bt_results = backtester.run(final_test_actual, final_test_pred, final_test_dates)

## 6. Future Prediction

In [ ]:
# Future Prediction (Ensemble Average, denormalized + clipped)
import numpy as np
import pandas as pd

norm_features_final = final_scaler.transform(features)
last_seq = np.expand_dims(norm_features_final[-config.SEQ_LENGTH:], axis=0)

future_preds_norm = [m.predict(last_seq, verbose=0) for m in final_ensemble_models]
avg_future = np.mean(future_preds_norm, axis=0)
future_returns = np.clip(final_target_scaler.inverse_transform(avg_future).flatten(), -0.2, 0.2)

# Convert returns to prices (aligned close, not pre-dropna frame)
last_price = float(pipeline.aligned_close[-1])
future_prices = []
p = last_price
for r in future_returns:
    p = p * (1 + float(r))
    future_prices.append(p)

future_dates = pd.bdate_range(start=pipeline.aligned_dates.iloc[-1], periods=config.FUTURE_DAYS + 1)[1:]

# Display
future_df = pd.DataFrame({'date': future_dates, 'predicted_price': future_prices, 'predicted_return': future_returns})
print(f"\n Last actual price: {last_price:.2f}")
display(future_df)


In [ ]:
# Final Plot
from modules.trainer import ModelTrainer
final_trainer = ModelTrainer(final_ensemble_models[0], config, final_scaler, CSV_PATH, target_scaler=final_target_scaler)
final_trainer.plot_predictions(
    y_true=final_test_actual,
    y_pred=final_test_pred,
    test_dates=final_test_dates,
    future_predictions=future_prices,
    future_dates=future_dates
)
import matplotlib.pyplot as plt
plt.show()


## 7. Model Save & Download

In [ ]:
# Save models (shared implementation adds dataset name + timestamp)
pipeline.save_models()
print("Future predictions + plot:")
pipeline.run_backtest()
pipeline.generate_final_prediction()
